In [3]:
import nltk
import re
import time
import tarfile
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import tarfile
import os
import re

def cargar_corpus_grande(ruta_tar, max_docs=150):
    corpus_grande = {}
    documentos_validos = 0
    
    if not os.path.exists(ruta_tar):
        print(f"Error: El archivo {ruta_tar} no se encuentra.")
        return corpus_grande

    with tarfile.open(ruta_tar, 'r') as tar:
        for member in tar:
            if documentos_validos >= max_docs:
                break
            
            if member.isfile() and member.name.endswith('.txt'):
                f = tar.extractfile(member)
                if f:
                    try:
                        # Lectura directa de los textos
                        texto = f.read().decode('utf-8', errors='ignore')
                        tokens = re.findall(r'\b[a-z]+\b', texto.lower())
                        if tokens:
                            corpus_grande[member.name] = tokens
                            documentos_validos += 1
                    except Exception:
                        continue
                        
    return corpus_grande

# Ejecución
ruta_tar = 'txt-files.tar' # Verifica que el nombre y la ruta coincidan con tu archivo extraído
corpus_grande = cargar_corpus_grande(ruta_tar, max_docs=150)
print(f"Cantidad de documentos válidos extraídos: {len(corpus_grande)}")

Cantidad de documentos válidos extraídos: 150


In [5]:
import numpy as np
import math

def construir_matrices_tfidf(corpus):
    N = len(corpus)
    if N == 0:
        return None, None, None, None

    # 1. Construir el vocabulario global y calcular DF (Document Frequency)
    df = {}
    for tokens in corpus.values():
        terminos_unicos = set(tokens)
        for termino in terminos_unicos:
            df[termino] = df.get(termino, 0) + 1
            
    vocabulario = list(df.keys())
    vocab_index = {termino: i for i, termino in enumerate(vocabulario)}
    num_terminos = len(vocabulario)
    
    # 2. Matriz TF (Term Frequency) - Matriz densa de documentos x términos
    tf_matrix = np.zeros((N, num_terminos))
    doc_nombres = list(corpus.keys())
    
    for i, doc_name in enumerate(doc_nombres):
        for termino in corpus[doc_name]:
            j = vocab_index[termino]
            tf_matrix[i, j] += 1
            
    # 3. Vector IDF aplicando la fórmula: log(N / df(t)) + 1
    idf_vector = np.zeros(num_terminos)
    for termino, count in df.items():
        j = vocab_index[termino]
        # Se calcula el logaritmo natural (base e) de N/df y se suma 1
        idf_vector[j] = math.log(N / count) + 1 
        
    # 4. Matriz ponderada TF-IDF (multiplicación por broadcasting de numpy)
    tf_idf_matrix = tf_matrix * idf_vector
    
    return tf_matrix, idf_vector, tf_idf_matrix, vocabulario

# Ejecución sobre el corpus cargado previamente
tf_mat, idf_vec, tfidf_mat, vocab = construir_matrices_tfidf(corpus_grande)

if tf_mat is not None:
    print(f"Dimensión de la matriz TF: {tf_mat.shape} (Documentos x Términos)")
    print(f"Muestra de vector IDF (primeros 5 términos): {idf_vec[:10]}")
    print(f"Dimensión de la matriz TF-IDF: {tfidf_mat.shape} (Documentos x Términos)")
    print(f"Vocabulario (primeros 10 términos): {vocab[:10]}")

Dimensión de la matriz TF: (150, 133352) (Documentos x Términos)
Muestra de vector IDF (primeros 5 términos): [1.60391605 1.         1.         1.         1.07616136 1.
 1.         1.         1.         1.        ]
Dimensión de la matriz TF-IDF: (150, 133352) (Documentos x Términos)
Vocabulario (primeros 10 términos): ['resolved', 'format', 'defective', 'actual', 'done', 'contract', 'often', 'required', 'prominently', 'regulating']


In [7]:
import pandas as pd

# 1. Extraer los nombres de los documentos respetando el orden de inserción
doc_nombres = list(corpus_grande.keys())

# 2. Recortar la matriz, los documentos y el vocabulario a los primeros 10 elementos
muestra_tf = tf_mat[:10, :10]
muestra_tfidf = tfidf_mat[:10, :10]
muestra_docs = doc_nombres[:10]
muestra_terminos = vocab[:10]

# 3. Crear el DataFrame asociando los datos con sus encabezados
df_muestra = pd.DataFrame(muestra_tf, index=muestra_docs, columns=muestra_terminos)
Tfidf_muestra = pd.DataFrame(muestra_tfidf, index=muestra_docs, columns=muestra_terminos)

print("Muestra de la matriz TF (10 documentos x 10 términos):")
# En Jupyter, 'display' genera una tabla interactiva mucho más legible que 'print'
display(df_muestra)

print("\nMuestra de la matriz TF-IDF (10 documentos x 10 términos):")
display(Tfidf_muestra)

Muestra de la matriz TF (10 documentos x 10 términos):


,resolved,format,defective,actual,done,contract,often,required,prominently,regulating
cache/epub/10000/pg10000.txt,2.0,4.0,3.0,1.0,9.0,1.0,1.0,3.0,2.0,1.0
cache/epub/10001/pg10001.txt,0.0,4.0,3.0,1.0,7.0,1.0,2.0,1.0,2.0,1.0
cache/epub/10002/pg10002.txt,8.0,4.0,3.0,1.0,12.0,3.0,5.0,2.0,2.0,1.0
cache/epub/10003/pg10003.txt,0.0,4.0,3.0,7.0,23.0,1.0,81.0,2.0,3.0,1.0
cache/epub/10004/pg10004.txt,0.0,4.0,3.0,2.0,30.0,1.0,17.0,2.0,2.0,1.0
cache/epub/10005/pg10005.txt,3.0,4.0,3.0,2.0,22.0,1.0,41.0,13.0,2.0,1.0
cache/epub/10006/pg10006.txt,2.0,4.0,3.0,2.0,4.0,1.0,13.0,3.0,2.0,1.0
cache/epub/10007/pg10007.txt,2.0,4.0,3.0,1.0,7.0,1.0,14.0,1.0,2.0,1.0
cache/epub/10008/pg10008.txt,4.0,4.0,3.0,2.0,22.0,1.0,13.0,4.0,2.0,1.0
cache/epub/10009/pg10009.txt,2.0,4.0,3.0,4.0,16.0,2.0,14.0,5.0,2.0,1.0



Muestra de la matriz TF-IDF (10 documentos x 10 términos):


,resolved,format,defective,actual,done,contract,often,required,prominently,regulating
cache/epub/10000/pg10000.txt,3.207832,4.0,3.0,1.0,9.685452,1.0,1.0,3.0,2.0,1.0
cache/epub/10001/pg10001.txt,0.000000,4.0,3.0,1.0,7.533130,1.0,2.0,1.0,2.0,1.0
cache/epub/10002/pg10002.txt,12.831328,4.0,3.0,1.0,12.913936,3.0,5.0,2.0,2.0,1.0
cache/epub/10003/pg10003.txt,0.000000,4.0,3.0,7.0,24.751711,1.0,81.0,2.0,3.0,1.0
cache/epub/10004/pg10004.txt,0.000000,4.0,3.0,2.0,32.284841,1.0,17.0,2.0,2.0,1.0
cache/epub/10005/pg10005.txt,4.811748,4.0,3.0,2.0,23.675550,1.0,41.0,13.0,2.0,1.0
cache/epub/10006/pg10006.txt,3.207832,4.0,3.0,2.0,4.304645,1.0,13.0,3.0,2.0,1.0
cache/epub/10007/pg10007.txt,3.207832,4.0,3.0,1.0,7.533130,1.0,14.0,1.0,2.0,1.0
cache/epub/10008/pg10008.txt,6.415664,4.0,3.0,2.0,23.675550,1.0,13.0,4.0,2.0,1.0
cache/epub/10009/pg10009.txt,3.207832,4.0,3.0,4.0,17.218582,2.0,14.0,5.0,2.0,1.0
